In [ ]:
import time
from tqdm import tqdm
import logging
import sys
import config
from data_process_helper import extract_card_data, download_card_image

In [2]:
logging.basicConfig(
    level="INFO",
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [3]:
json_files = list(config.VALIDATION_SOURCE_PATH.glob("*.json"))
df = extract_card_data(json_files)
df.count()

100%|██████████| 8/8 [00:00<00:00, 32.86it/s]


layout     5315
name       5315
setCode    5315
number     5315
dtype: int64

In [6]:

df_clean = df[df['setCode'].notna() & df['number'].notna()].copy()
df_clean['card_id'] = df_clean['setCode'] + '-' + df_clean['number'].astype(str)
    
# Remove duplicates based on card_id
df_clean = df_clean.drop_duplicates(subset=['card_id'], keep='first')

config.VALIDATION_IMAGE_PATH.mkdir(exist_ok=True)

# Get already downloaded images
existing_files = config.VALIDATION_IMAGE_PATH.glob("*.jpg")
existing_ids = {f.stem for f in existing_files}
    
# Filter out already downloaded cards
df_to_download = df_clean[~df_clean['card_id'].isin(existing_ids)]
    
logger.info(f"Total cards: {len(df_clean)}")
logger.info(f"Already downloaded: {len(existing_ids)}")
logger.info(f"To download: {len(df_to_download)}")
    
if len(df_to_download) > 0:
    success = 0
    failed = 0
        
    pbar = tqdm(df_to_download.iterrows(), total=len(df_to_download), desc="Downloading cards")
        
    for _, row in pbar:
        set_code = row['setCode'].lower()
        number = row['number']
        card_id = row['card_id']

        file_path = config.VALIDATION_IMAGE_PATH / f"{card_id}.jpg"
        if download_card_image(set_code, number, file_path):
            success +=1
        else:
            failed +=1
        time.sleep(0.1) # sleep for 100ms
        pbar.set_postfix(success=success, failed=failed)


2025-10-03 02:34:48 - INFO - Total cards: 5050
2025-10-03 02:34:48 - INFO - Already downloaded: 5048
2025-10-03 02:34:48 - INFO - To download: 2


In [5]:
df.to_parquet(config.VALIDATION_DATA_FILE_PATH, engine='pyarrow', compression='snappy', index=False)
df.describe()

,layout,name,setCode,number
count,5315,5315,5315,5315
unique,13,3928,8,3461
top,normal,Plains,PRM,43
freq,4708,42,3225,9
